In [24]:
USERNAME_elastic = "UDAP_Elastic"
MDP_elastic = "F*rge2024!"
ENDPOINT = "https://noeyyalp.noe.edf.fr:29203"


In [2]:
from pathlib import Path
import sys

# Add the project root directory to sys.path
sys.path.append(str(Path().resolve().parent))
from src.constants.paths import SECRET_PATH

SECRET_PATH

from src.processing.pde_ple import PDE, PLE, es
from src.processing.document import Document
from src.constants.constants import S3_DOCS_PATH


/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/elasticsearch/_sync/client/__init__.py:397: SecurityWarning: Connecting to 'https://noeyyalp.noe.edf.fr:29203' using TLS with verify_certs=False is insecure
  _transport = transport_class(


In [3]:
doc = Document(
    s3_path=S3_DOCS_PATH
    + "Services.004/G-Environnement.007/02 - Réglementation/04 - HSE Compliance/07 - Copie HSE Compliance AFA/ENVIRONNEMENT/export-1000000083-risques-risks-120118.xls"
)

In [5]:
doc.download()

In [11]:
import os
import subprocess
tempdir = os.path.dirname(doc.local_path)

In [13]:
doc.local_path


'/opt/app-root/src/uc202-ipn-rex/temp_dir/export-1000000083-risques-risks-120118.xls'

In [9]:
docs = loader.load()


KeyboardInterrupt: 

In [11]:
import re
def remove_xml_tags(text):
    # Use regular expression to remove XML tags
    clean_text = re.sub(r"<[^>]+>", "", text)
    return clean_text

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def precision_at_k(question_embeddings, answer_embeddings, labels, k=5):
    """
    question_embeddings : embeddings des questions (split test)
    answer_embeddings : embeddings de toutes les réponses possibles
    labels : labels des réponses (0=pertinent, 1=non pertinent)
    k : nombre de résultats à considérer
    """
    precisions = []
    for q_emb in question_embeddings:
        # Calculer les similarités entre la question et toutes les réponses
        similarities = cosine_similarity([q_emb], answer_embeddings)[0]
        # Trier les indices des réponses par similarité décroissante
        sorted_indices = np.argsort(similarities)[::-1]
        # Prendre les k premières réponses
        top_k_indices = sorted_indices[:k]
        # Compter le nombre de réponses pertinentes (label=0) dans le top k
        relevant = sum(labels[i] == 0 for i in top_k_indices)
        precisions.append(relevant / k)
    # Retourner la moyenne des Precision@k
    return np.mean(precisions)


In [12]:
def read_and_clean_file(file_path):
    # Read the file content
    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()

    # Clean the content
    cleaned_content = remove_xml_tags(content)
    return cleaned_content


In [13]:
file_path = '/opt/app-root/src/uc202-ipn-rex/temp_dir/export-1000000083-risques-risks-120118.xls.txt'
cleaned_text = read_and_clean_file(file_path)


In [14]:
cleaned_text


"Sheet: export-1000000083-risques-risks  Unnamed: 1 Unnamed: 2 Unnamed: 3 Unnamed: 4 Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 8 Unnamed: 9 Unnamed: 10 Unnamed: 11 Unnamed: 12 Unnamed: 13 Unnamed: 14 Unnamed: 15 Unnamed: 16 Unnamed: 17 Unnamed: 18 Unnamed: 19 Unnamed: 20 Unnamed: 21 Unnamed: 22 Unnamed: 23 Unnamed: 24 Unnamed: 25 Unnamed: 26 Unnamed: 27 Unnamed: 28 Unnamed: 29 Unnamed: 30 Unnamed: 31 Unnamed: 32 Unnamed: 33 Unnamed: 34 Unnamed: 35 Unnamed: 36 Unnamed: 37 Unnamed: 38 Unnamed: 39 Unnamed: 40 Unnamed: 41 Unnamed: 42 Unnamed: 43 Unnamed: 44 Unnamed: 45 Unnamed: 46 Unnamed: 47 Unnamed: 48 Unnamed: 49 Unnamed: 50 Unnamed: 51 Unnamed: 52 Unnamed: 53 Unnamed: 54 Unnamed: 55 Unnamed: 56 Unnamed: 57 Unnamed: 58 Unnamed: 59 Unnamed: 60 Unnamed: 61 Unnamed: 62 Unnamed: 63 Unnamed: 64 Unnamed: 65 Unnamed: 66 Unnamed: 67 Unnamed: 68 Unnamed: 69 Unnamed: 70 Unnamed: 71 Unnamed: 72 Unnamed: 73 Unnamed: 74 Unnamed: 75 Unnamed: 76 Unnamed: 77 Unnamed: 78 Unnamed: 79 Unnamed: 80 Unnamed:

In [ ]:
            
            subprocess.run(
                [
                    "libreoffice",
                    "--headless",
                    "--convert-to",
                    "xlsx",
                    doc.local_path,
                    "--outdir",
                    tempdir
                ],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
                check=True,
            )

CompletedProcess(args=['libreoffice', '--headless', '--convert-to', 'xlsx', '/opt/app-root/src/uc202-ipn-rex/temp_dir/export-1000000083-risques-risks-120118.xls', '--outdir', '/opt/app-root/src/uc202-ipn-rex/temp_dir'], returncode=0)

In [20]:
import pandas as pd

# Read the Excel file
df = pd.read_excel(doc.local_path, engine="openpyxl")


# Extract text from all cells
text = df.to_string(index=False, header=False)

print(text)


BadZipFile: File is not a zip file

In [22]:
import pymupdf4llm
text = pymupdf4llm.to_markdown(doc.local_path)

FileDataError: Failed to open file '/opt/app-root/src/uc202-ipn-rex/temp_dir/export-1000000083-risques-risks-120118.xls'.

In [32]:
import polars as pl


def get_all_file_paths(root_path):
    useful_extensions = {
        ".pdf",
        ".doc",
        ".docx",
        ".txt",
        ".rtf",
        ".odt",
        ".ppt",
        ".pptx",
        ".pps",
        ".ppsx",
        ".odp",
        ".xls",
        ".xlsx",
        ".xlsb",
        ".ods",
        ".csv",
    }

    seen_paths = set()
    all_file_paths = set()
    file_count = 0

    stack = [(root_path, 0)]

    while stack:
        current_path, depth = stack.pop()
        if depth > 40 or current_path in seen_paths:
            continue
        seen_paths.add(current_path)

        try:
            df = PLE.s3.ls(current_path)
            paths = set(df["path"].to_list())  # deduplication at this level
        except Exception as e:
            print(f"Error listing {current_path}: {e}")
            continue

        for path in paths:
            file_name = path.split("/")[-1]
            if "." in file_name:
                ext = "." + file_name.split(".")[-1].lower()
                if ext in useful_extensions:
                    if path not in all_file_paths:
                        all_file_paths.add(path)
                        file_count += 1
                        if file_count % 2000 == 0:
                            print(f"Reached {file_count} files")
                continue
            else:
                # Assume it's a directory
                stack.append((path, depth + 1))

    return ["s3://" + p for p in all_file_paths]


root_path = (
    "s3://bkt-pud-uc/uc202-rex/FA3_useful_files_final/Services.004/G-Environnement.007"
)
file_paths = get_all_file_paths(root_path)
file_paths = [s for s in file_paths]


Reached 2000 files
Reached 4000 files
Reached 6000 files
Reached 8000 files


In [ ]:
import chardet
def detect_encoding(file_path):
    with open(file_path, "rb") as f:
        result = chardet.detect(f.read())
    return result["encoding"]
detect_encoding("/opt/app-root/src/uc202-ipn-rex/G-Environnement.007.csv")

'Windows-1252'

In [45]:
import pandas as pd
df = pd.read_csv(
    "/opt/app-root/src/uc202-ipn-rex/G-Environnement.007.csv", encoding="Windows-1252", sep=";",header=1
)

In [62]:
's3://'+PLE.s3.ls(file_paths[0])['path'][0]==file_paths[0]

True

In [90]:
file_paths=PDE.s3.list_objects(
    "s3://bkt-pud-uc/uc202-rex/FA3_useful_files_final/Services.004/G-Environnement.007"
)

In [53]:
file_paths_local=list(df["Chemin d'accès"])
prefix = "Q:\\CO\\DIPNN-FLA3\\Services.004\\G-Environnement.007\\"
cleaned_local_paths = [path.replace(prefix, "").replace("\\",'/') for path in file_paths_local]
cleaned_local_paths

['ECFA070459.docx',
 'Gestion bouteilles de gaz.pptx',
 'Réchauffement Climatique-Transports.docx',
 'S24-2018 stat AFA.xlsx',
 'Sensibilisation Enviro juin 2019 fusion .ppt',
 'Suivi PME Avril-Mai.ppt',
 '01 - Réseau HSE - suivi titulaires/07 - Audits aux etps - Raccourci.lnk',
 '01 - Réseau HSE - suivi titulaires/Analyses environnementales - Raccourci.lnk',
 '01 - Réseau HSE - suivi titulaires/01 - Suivi des documents/01 - Tableau de suivi/Courriers transmis aux etps/D458517040153 [ ] [ ].pdf',
 '01 - Réseau HSE - suivi titulaires/01 - Suivi des documents/01 - Tableau de suivi/Courriers transmis aux etps/D458517040167 [ ] [ ].pdf',
 '01 - Réseau HSE - suivi titulaires/01 - Suivi des documents/01 - Tableau de suivi/Courriers transmis aux etps/D458517040173 [ ] [ ].pdf',
 '01 - Réseau HSE - suivi titulaires/01 - Suivi des documents/02 - Trame AE/Trame Analyse Environnementale_Ind 5.docx',
 '01 - Réseau HSE - suivi titulaires/01 - Suivi des documents/02 - Trame AE/Trame Analyse Environn

In [91]:
prefix = "s3://bkt-pud-uc/uc202-rex/FA3_useful_files_final/Services.004/G-Environnement.007/"
cleaned_s3_paths = [path.replace(prefix, "") for path in file_paths]
cleaned_s3_paths
cleaned_s3_paths

['01 - Réseau HSE - suivi titulaires/01 - Suivi des documents/01 - Tableau de suivi/Courriers transmis aux etps/D458517040153 [ ] [ ].pdf',
 '01 - Réseau HSE - suivi titulaires/01 - Suivi des documents/01 - Tableau de suivi/Courriers transmis aux etps/D458517040167 [ ] [ ].pdf',
 '01 - Réseau HSE - suivi titulaires/01 - Suivi des documents/01 - Tableau de suivi/Courriers transmis aux etps/D458517040173 [ ] [ ].pdf',
 '01 - Réseau HSE - suivi titulaires/01 - Suivi des documents/01 - Tableau de suivi/~$GESTION DOCUMENTAIRE 2018.xlsx',
 '01 - Réseau HSE - suivi titulaires/01 - Suivi des documents/02 - Trame AE/Trame Analyse Environnementale_Ind 5.docx',
 '01 - Réseau HSE - suivi titulaires/01 - Suivi des documents/02 - Trame AE/Trame Analyse Environnementale_Ind 6.docx',
 '01 - Réseau HSE - suivi titulaires/01 - Suivi des documents/03 - Contrôles de la réalisation des AE/2017/Cartographie_CCS et PdS actifs V2017.xlsx',
 '01 - Réseau HSE - suivi titulaires/01 - Suivi des documents/03 - Con

In [99]:
useful_extensions = {
    ".pdf",
    ".doc",
    ".docx",
    ".txt",
    ".rtf",
    ".odt",
    ".ppt",
    ".pptx",
    ".pps",
    ".ppsx",
    ".odp",
    ".xls",
    ".xlsx",
    ".xlsb",
    ".ods",
    ".csv",
}


In [102]:
ch = "01 - Réseau HSE - suivi titulaires/01 - Suivi des documents/01 - Tableau de suivi/Courriers transmis aux etps/D458517040153 [ ] [ ].pdf"
ch.split(".")[-1].lower()

'pdf'

In [104]:
not_cleaned_s3_paths = [
    p for p in cleaned_s3_paths if '.'+p.split(".")[-1].lower() not in useful_extensions
]
len(not_cleaned_s3_paths)

1745

In [105]:
len(cleaned_s3_paths) - len(not_cleaned_s3_paths)

11135

In [1]:
import docx